# Arabic Media Bias Detection — ARBERT Fine-tuning

**Runtime**: Google Colab (T4 GPU recommended)  
**Model**: UBC-NLP/ARBERTv2  
**Task**: Binary classification — Non-biased (0) vs Biased (1)

---

### Quick-start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells top to bottom (`Runtime → Run all`)
3. Choose your experiment in **Cell 4**

No Google Drive required. Everything runs inside this Colab instance.

## Cell 1 — GPU check

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if device == 'cuda':
    print(f'✅ GPU ready: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU detected.')
    print('   Training will work but will be slow (~30 min/epoch).')
    print('   Recommended: Runtime → Change runtime type → T4 GPU')

## Cell 2 — Install dependencies

In [ ]:
# Install required packages.
# NOTE: You may see a warning about fsspec version conflict with gcsfs.
# This is a pre-existing Colab environment conflict — it does NOT affect
# this project. The pipeline does not use gcsfs or Google Cloud Storage.

print('Installing packages...')
import subprocess, sys

packages = [
    'transformers==4.44.2',
    'accelerate>=0.30.0',
    'evaluate>=0.4.0',
    'scikit-learn>=1.3.0',
    'pandas>=2.0.0',
    'seaborn>=0.12.0',
    'pyyaml>=6.0',
]

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    print('❌ pip install failed:')
    print(result.stderr[-3000:])
    raise RuntimeError('Package installation failed. See error above.')

for line in result.stderr.splitlines():
    if 'fsspec' not in line and line.strip():
        print(line)

print('✅ All packages installed.')

## Cell 3 — Clone repository

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/Abjasser/arabic-bias-detection.git'
REPO_DIR = 'arabic-bias-detection'

# Always start from Colab root to avoid nested chdir issues
os.chdir('/content')

if Path(REPO_DIR).is_dir():
    print('Repository already exists — pulling latest changes...')
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull'],
        capture_output=True, text=True
    )
else:
    print(f'Cloning {REPO_URL} ...')
    result = subprocess.run(
        ['git', 'clone', REPO_URL],
        capture_output=True, text=True
    )

if result.returncode != 0:
    print('❌ Git operation failed:')
    print(result.stderr)
    raise RuntimeError(
        'Could not clone the repository.\n'
        'Possible causes:\n'
        '  • No internet connection in this Colab session\n'
        '  • Repository URL has changed\n'
        f'URL tried: {REPO_URL}'
    )

print(result.stdout.strip() or result.stderr.strip())

if not Path(REPO_DIR).is_dir():
    raise RuntimeError(f'Expected directory "{REPO_DIR}" not found after clone.')

os.chdir(REPO_DIR)

repo_root = str(Path.cwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f'✅ Working directory: {Path.cwd()}')
print(f'   Files: {[f.name for f in Path(".").iterdir() if not f.name.startswith(".")]}')

## Cell 4 — Choose experiment

In [ ]:
# ── Select one experiment ──────────────────────────────────────────────────
# EXPERIMENT = 'configs/baseline_gpt_only.yaml'     # Experiment A: GPT-only
EXPERIMENT  = 'configs/main_gpt_claude_mixed.yaml'  # Experiment B: GPT + Claude  ← DEFAULT
# ──────────────────────────────────────────────────────────────────────────

from pathlib import Path

if not Path(EXPERIMENT).exists():
    raise FileNotFoundError(
        f'Config not found: {EXPERIMENT}\n'
        'Make sure Cell 3 (clone) ran successfully.'
    )

print(f'✅ Experiment config: {EXPERIMENT}')

from src.utils import load_config
cfg = load_config(EXPERIMENT)
print(f'   Name    : {cfg["experiment_name"]}')
print(f'   Model   : {cfg["model"]["name"]}')
print(f'   Sources : {cfg["data"]["sources_to_include"]}')
print(f'   Epochs  : {cfg["training"]["epochs"]}')

## Cell 5 — Prepare data

In [ ]:
import subprocess, sys
from pathlib import Path

print('Preparing dataset...')
result = subprocess.run(
    [sys.executable, 'src/prepare_data.py', '--config', EXPERIMENT],
    capture_output=False
)

if result.returncode != 0:
    raise RuntimeError('prepare_data.py failed. Check the output above.')

from src.utils import load_config
cfg = load_config(EXPERIMENT)
for label, path_key in [('Train CSV', 'train_path'), ('Test CSV', 'test_path')]:
    p = Path(cfg['data'][path_key])
    if not p.exists():
        raise FileNotFoundError(f'{label} not created at: {p}')
    print(f'✅ {label}: {p}  ({p.stat().st_size // 1024} KB)')

## Cell 6 — Data preview

In [ ]:
import json
import pandas as pd
from pathlib import Path
from src.utils import load_config

cfg      = load_config(EXPERIMENT)
train_df = pd.read_csv(cfg['data']['train_path'])
test_df  = pd.read_csv(cfg['data']['test_path'])

print(f"Experiment : {cfg['experiment_name']}")
print(f"Train rows : {len(train_df):,}")
print(f"Test  rows : {len(test_df):,}")
print(f"\nTrain label distribution:")
print(train_df['label'].value_counts().rename({0: 'Non-biased', 1: 'Biased'}).to_string())
if 'source' in train_df.columns:
    print(f"\nTrain source distribution:")
    print(train_df['source'].value_counts().to_string())
print(f"\nSample rows:")
display(train_df[['arabic_text','label','source']].head(3))

## Cell 7 — Train  *(~15–25 min on T4)*

In [ ]:
import subprocess, sys
from pathlib import Path
from src.utils import load_config

print('Starting training...')
print('(This cell streams output live. Grab a coffee ☕)')
print()

result = subprocess.run(
    [sys.executable, 'src/train.py', '--config', EXPERIMENT],
    capture_output=False
)

if result.returncode != 0:
    raise RuntimeError('train.py failed. Check the output above.')

cfg = load_config(EXPERIMENT)
best_model   = Path(cfg['output_dir']) / 'best_model'
final_model  = Path('outputs') / 'final_model'

for label, path in [('best_model', best_model), ('final_model', final_model)]:
    if not path.exists():
        raise FileNotFoundError(
            f'{label} not found at {path}.\n'
            'Training may have completed but the model was not saved correctly.'
        )
    print(f'✅ {label}: {path}')

print(f'\n   final_model contents: {[p.name for p in final_model.iterdir()]}')

## Cell 7a — Verify final model loads correctly

In [ ]:
import torch, json
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

FINAL_MODEL = Path('outputs/final_model')

if not FINAL_MODEL.exists():
    raise FileNotFoundError(
        f'outputs/final_model not found.\n'
        'Make sure Cell 7 (training) completed successfully.'
    )

print('Verifying final_model can be reloaded from disk...')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(str(FINAL_MODEL))
model     = AutoModelForSequenceClassification.from_pretrained(str(FINAL_MODEL)).to(device)
model.eval()

print(f'✅ Tokenizer loaded  | vocab size: {tokenizer.vocab_size:,}')
print(f'✅ Model loaded      | labels: {model.config.id2label}')
print(f'   Parameters       : {sum(p.numel() for p in model.parameters()):,}')

# Quick smoke-test: classify one Arabic sentence
test_text = 'أكد المسؤول نجاح السياسة رغم الأرقام التي تشير إلى عكس ذلك.'
enc = tokenizer(test_text, return_tensors='pt', truncation=True,
                padding='max_length', max_length=128).to(device)
with torch.no_grad():
    logits = model(**enc).logits
pred = model.config.id2label[torch.argmax(logits, dim=-1).item()]
conf = torch.softmax(logits, dim=-1).max().item()

print(f'\n✅ Smoke-test passed')
print(f'   Input  : {test_text}')
print(f'   Pred   : {pred}  (confidence: {conf:.1%})')

# Show model card
card_path = FINAL_MODEL / 'model_card.json'
if card_path.exists():
    with open(card_path) as f:
        card = json.load(f)
    print(f'\nModel card:')
    for k, v in card.items():
        print(f'  {k}: {v}')

## Cell 7b — Zip and download final_model

In [ ]:
import zipfile, os, json
from pathlib import Path
from google.colab import files

FINAL_MODEL = Path('outputs/final_model')
ZIP_PATH    = Path('final_arabic_bias_model.zip')

if not FINAL_MODEL.exists():
    raise FileNotFoundError(
        'outputs/final_model not found.\n'
        'Run Cell 7 (training) and Cell 7a (verify) first.'
    )

# Remove any previous zip
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print(f'Zipping {FINAL_MODEL} ...')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(FINAL_MODEL.rglob('*')):
        if file_path.is_file():
            arcname = file_path.relative_to(FINAL_MODEL.parent)
            zf.write(file_path, arcname)
            size_mb = file_path.stat().st_size / (1024 * 1024)
            print(f'  + {arcname}  ({size_mb:.1f} MB)')

zip_size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
print(f'\n✅ Archive ready: {ZIP_PATH}  ({zip_size_mb:.1f} MB)')

print('Downloading...')
files.download(str(ZIP_PATH))
print(f'⬇️  {ZIP_PATH.name}  ({zip_size_mb:.1f} MB)')
print()
print('To reload the model locally:')
print('  from transformers import AutoTokenizer, AutoModelForSequenceClassification')
print('  tokenizer = AutoTokenizer.from_pretrained("final_model")')
print('  model     = AutoModelForSequenceClassification.from_pretrained("final_model")')

## Cell 8 — Evaluate

In [ ]:
import subprocess, sys

print('Running evaluation...')
result = subprocess.run(
    [sys.executable, 'src/evaluate.py', '--config', EXPERIMENT],
    capture_output=False
)

if result.returncode != 0:
    raise RuntimeError('evaluate.py failed. Check the output above.')

print('\n✅ Evaluation complete.')

## Cell 9 — Results summary

In [ ]:
import json
from pathlib import Path
from src.utils import load_config

cfg          = load_config(EXPERIMENT)
metrics_path = Path(cfg['output_dir']) / 'metrics.json'

if not metrics_path.exists():
    raise FileNotFoundError(f'metrics.json not found. Did evaluate.py complete?')

with open(metrics_path) as f:
    m = json.load(f)

print(f"\n{'='*45}")
print(f"  Results: {m['experiment']}")
print(f"{'='*45}")
print(f"  Accuracy        : {m['accuracy']:.4f}  ({m['accuracy']*100:.1f}%)")
print(f"  F1 Macro        : {m['f1_macro']:.4f}")
print(f"  F1 Weighted     : {m['f1_weighted']:.4f}")
print(f"  Precision Macro : {m['precision_macro']:.4f}")
print(f"  Recall Macro    : {m['recall_macro']:.4f}")
print(f"  ─────────────────────────────────────────")
print(f"  F1  Non-biased  : {m['f1_non_biased']:.4f}")
print(f"  F1  Biased      : {m['f1_biased']:.4f}")
print(f"  Test rows       : {m['test_rows']}")

In [ ]:
from IPython.display import Image, display
from pathlib import Path
from src.utils import load_config

cfg     = load_config(EXPERIMENT)
cm_path = Path(cfg['output_dir']) / 'confusion_matrix.png'

if cm_path.exists():
    display(Image(str(cm_path)))
else:
    print('⚠️  confusion_matrix.png not found — did evaluate.py run?')

## Cell 10 — Inference on new sentences

In [ ]:
import subprocess, sys, tempfile, os
from pathlib import Path

test_sentences = [
    'أعلنت الحكومة عن خطة اقتصادية جديدة لتحسين مستوى المعيشة.',
    'يسعى الإعلام المتحيز دائماً إلى تشويه الحقائق لخدمة أجندات خاصة.',
    'التقرير يستعرض الأرقام الرسمية دون تعليق.',
]

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt',
                                  delete=False, encoding='utf-8') as f:
    f.write('\n'.join(test_sentences))
    tmp_path = f.name

result = subprocess.run(
    [sys.executable, 'src/predict.py',
     '--config', EXPERIMENT,
     '--input', tmp_path],
    capture_output=True, text=True
)

os.unlink(tmp_path)

if result.returncode != 0:
    print('❌ predict.py failed:')
    print(result.stderr)
else:
    print(result.stdout)

## Cell 11 — Download all output files

In [ ]:
from google.colab import files
from pathlib import Path
from src.utils import load_config

cfg = load_config(EXPERIMENT)
out = Path(cfg['output_dir'])

to_download = [
    'metrics.json',
    'classification_report.txt',
    'confusion_matrix.png',
    'predictions.csv',
    'cross_source_metrics.json',
]

print('Downloading output files...')
for fname in to_download:
    p = out / fname
    if p.exists():
        files.download(str(p))
        print(f'  ⬇️  {fname}  ({p.stat().st_size // 1024} KB)')
    else:
        print(f'  —  {fname} (not present, skipping)')

# Also offer the zip again if it exists
zip_path = Path('final_arabic_bias_model.zip')
if zip_path.exists():
    print(f'\n  ⬇️  {zip_path.name}  ({zip_path.stat().st_size // (1024*1024)} MB) — re-downloading model zip')
    files.download(str(zip_path))

---
## Reference

| Setting | Value |
|---------|-------|
| Default model | `UBC-NLP/ARBERTv2` |
| Switch model | Edit `model.name` in the YAML config |
| Alternative | `aubmindlab/bert-base-arabertv2` |
| Epochs | 5 — edit `training.epochs` in YAML |
| Best checkpoint | Selected by F1-macro |
| Leakage guard | Group-aware split on `sentence_id` |
| Final model path | `outputs/final_model/` |
| Model zip | `final_arabic_bias_model.zip` |

**Experiment A** (`baseline_gpt_only`): 725 GPT-translated sentences, single-source baseline.  
**Experiment B** (`main_gpt_claude_mixed`): 1 468 sentences (GPT + Claude), primary robust experiment.

**To reload the downloaded model locally:**
```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
# Unzip final_arabic_bias_model.zip first, then:
tokenizer = AutoTokenizer.from_pretrained('final_model')
model     = AutoModelForSequenceClassification.from_pretrained('final_model')
```